## Grid 1 \u2014 Clinical Variable Distributions

Five individual charts for the key clinical variables across all 457 patients.

- **Continuous** (age, bmi, height, weight): 30-bin histogram with a dashed red mean line.
- **Ordinal** (asa): bar chart showing patient count per ASA class.

Data source: `data/processed/clinical_lab_updated.csv`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# Load data - path is relative to this notebook's location (notebooks/eda/).
data_path = os.path.join('..', '..', 'data', 'processed', 'clinical_lab_updated.csv')
df = pd.read_csv(data_path)

# Report how many non-null values each variable has.
print(f'Loaded {len(df)} rows.')
for col in ['age', 'bmi', 'height', 'weight', 'asa']:
    print(f'  {col}: {df[col].notna().sum()} non-null values')

# ---- Shared style helper --------------------------------------------------------
def style_ax(ax):
    """
    Apply consistent clean styling to any axis:
    - Remove top and right spines (the box frame looks heavy in research figures)
    - Add a solid hairline y-axis grid (recessive, not dashed -- dashed reads as
      a threshold/projection rather than a grid)
    - Keep left and bottom spines only
    """
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, color='#cccccc', linewidth=0.8, linestyle='-')
    ax.set_axisbelow(True)   # grid draws behind the bars, not on top of them

# ---- Helper: continuous histogram -----------------------------------------------
def plot_hist(series, title, xlabel, filename):
    """
    Creates a standalone 30-bin histogram for one continuous variable.
    Saves to the current working directory at 300 dpi.
    """
    clean = series.dropna()          # remove NaN before counting or plotting
    n = len(clean)

    fig, ax = plt.subplots(figsize=(8, 5))

    # steelblue bars; white edge creates a small gap between adjacent bars
    ax.hist(clean, bins=30, color='steelblue', edgecolor='white', linewidth=0.6)

    # Vertical dashed red line at the mean
    mean_val = clean.mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
               label=f'mean = {mean_val:.1f}')

    ax.set_title(f'{title} (n={n})', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=10, framealpha=0.7)

    style_ax(ax)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f'Saved -> {filename}')
    plt.show()

# ---- Helper: ordinal bar chart --------------------------------------------------
def plot_bar(series, title, filename):
    """
    Creates a standalone bar chart for an ordinal variable (ASA class).
    Bars sorted by ASA class value (1 -> 5). No mean line (not meaningful for ordinal).
    """
    clean = series.dropna()
    n = len(clean)

    # sort_index() keeps ASA 1, 2, 3, 4, 5 in natural order
    counts = clean.value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(counts.index.astype(str), counts.values,
           color='steelblue', edgecolor='white', linewidth=0.6)

    ax.set_title(f'{title} (n={n})', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('ASA Class', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)

    style_ax(ax)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f'Saved -> {filename}')
    plt.show()

# ---- Plot and save each variable as its own figure ------------------------------
plot_hist(df['age'],    'Age',       'Age (years)',  'clinical_hist_age.png')
plot_hist(df['bmi'],    'BMI',       'BMI (kg/m\u00b2)',  'clinical_hist_bmi.png')
plot_hist(df['height'], 'Height',    'Height (cm)',  'clinical_hist_height.png')
plot_hist(df['weight'], 'Weight',    'Weight (kg)',  'clinical_hist_weight.png')
plot_bar( df['asa'],    'ASA Class',                 'clinical_hist_asa.png')